In [ ]:
from matplotlib import gridspec
import scipy.optimize
from scipy.special import factorial

# For auxilary function???

def normal(x, vals):
        std = vals[1]*x + 0.05
        return vals[2]*np.exp(-(1/2)*((x-vals[0])/std)**2)

def fitted_curve(x, vals):
    return vals[0]*(np.tanh(vals[1]*x) + np.e**(-vals[2]*x) - 1)

def gen_fitted_curve(vals):
    def fitted_curve(x):
        return vals[0]*(np.tanh(vals[1]*x) + np.e**(-vals[2]*x) - 1)
    return fitted_curve
    
def gen_norm(vals):
    def nor(x):
        std = vals[1]*x + 0.05
        return vals[2]*np.exp(-(1/2)*((x-vals[0])/std)**2)
    return nor

def poisson(x, vals):                  #vals is mu then coefficient
    x = x*vals[3] + vals[2]
    return vals[1]*np.exp(-vals[0])*(vals[0]**x)/factorial(x)

def gen_poi(vals):
    def poi(x):                  #vals is mu then coefficient
        x = x*vals[3] + vals[2]
        return vals[1]*np.exp(-vals[0])*(vals[0]**x)/factorial(x)
    return poi

def gen_chisq(model_funct):
    def chi(modelparams, x_data, y_data, y_err):
        chisqval=0
        for i in range(len(x_data)):
            chisqval += ((y_data[i] - model_funct(x_data[i], modelparams))/y_err[i])**2
        return chisqval
    return chi

labels = ['Particle number', 'Pairs', 'Diagonal Pairs', 'Seperated Pairs', 'Triangles', 'Chain - 3', 'Chain - 4']
    
chisq = gen_chisq(fitted_curve)

In [ ]:
def gen_pro(symm, parts, empty):
    def pro(lat_fil):
        return 2*symm*((1 - lat_fil)**empty)*(lat_fil**(parts - 1))
    return pro

In [ ]:
# Larger distribution constituents statistics 


lat_fils = np.linspace(0.001, 0.6, 200)

fig = plt.figure(figsize=(17, 5))
spec = gridspec.GridSpec(ncols=2, nrows=1,
                         width_ratios=[1, 1], wspace=0.3)
axes = fig.add_subplot(spec[0])
plt.yticks(fontsize=16)
plt.xticks(fontsize=16)
ax2 = fig.add_subplot(spec[1])
initial = np.array([0.5, 20, 15])
const_error = 0.01*np.ones(100)

inset_ax = axes.inset_axes(
   [0.25, 0.35, 0.7, 0.6],  # [x, y, width, height] w.r.t. axes
    xlim=[0, 0.4], ylim=[0, 0.2], yticklabels=[0, '', 0.05, '', 0.10, '', 0.15, '', 0.20] # sets viewport &amp; tells relation to main axes
    )

coefficients = []
normalization = np.zeros(100)

stop = 0
leftover = np.ones(200)

for geometry in orbits:
    symm = len(geometry)
    particles = geometry[0].sum()
    fy, fx = geometry[0].shape
    empty = (fy+2)*(fx+2) - particles
    
    pro = gen_pro(symm, particles, empty)
    prob_data = pro(lat_fils)
    if particles == 1:
        prob_data = prob_data/2
        stop = 1
    axes.plot(lat_fils, prob_data)
    if stop == 0:
        inset_ax.plot(lat_fils, prob_data)

    #ys, error = i, o
    # normalization = normalization + ys
    #axes.plot(lat_fils, ys)

    #for particles not counted
    leftover = leftover - prob_data
    
    # axes.fill_between(lat_fils, prob_data-error, prob_data+error, alpha=0.5) 
    # if stop == 0:
    #     inset_ax.fill_between(lat_fils, prob_data-error, prob_data+error, alpha=0.5)
    stop = 0
    #for i in range(5):
        #print(populations[:,1][100*i])
    if False:
        fit = scipy.optimize.minimize(chisq, initial, args=(lat_fils, ys, const_error))
        #print(fit.message)
        #print(fit.x)
        coefficients.append(fit.x)
    
        model = gen_fitted_curve(fit.x)
        #model = gen_norm(initial)
        #print(model(0.12))
        modelled_data = model(lat_fils)
    
        ax2.plot(lat_fils, modelled_data)
no = 'no'

symm = 1/2
particles = 1
fy, fx = 5, 5
empty = 24

pro = gen_pro(symm, particles, empty)
prob_data = pro(lat_fils)

leftover = leftover - prob_data
axes.plot(lat_fils, prob_data)
axes.indicate_inset_zoom(inset_ax, edgecolor="blue")

ax2.plot(lat_fils, leftover, color = 'black', label='Uncounted Particles')
# ax2.legend()
axes.set_xlabel(r"Lattice filling", fontsize=20)
ax2.set_xlabel(r"Lattice filling", fontsize=20)
axes.set_ylabel('Particle fraction found \n in configuration', fontsize=20)

ax2.set_ylabel('Particle fraction outside of all \n 3x3 configurations', fontsize=20)

axes.minorticks_on()
axes.tick_params(which='major', length=10, width=2, direction='in', bottom=True, top=True, left=True, right=True)
axes.tick_params(which='minor', length=5, width=2, direction='in', bottom=True, top=True, left=True, right=True)

ax2.minorticks_on()
ax2.tick_params(which='major', length=10, width=2, direction='in', bottom=True, top=True, left=True, right=True)
ax2.tick_params(which='minor', length=5, width=2, direction='in', bottom=True, top=True, left=True, right=True)
plt.yticks(fontsize=16)
plt.xticks(fontsize=16)
axes.set_ylim([0, 1])
ax2.set_ylim([0, 1])
axes.set_xlim([0, 0.6])
ax2.set_xlim([0, 0.6])

print(holds[22])
plt.savefig(f'LEFTOVERS.pdf', bbox_inches='tight')
plt.show(block=False)
print(leftover)

In [ ]:
# Other animations (oribt ramseys, anisotropic stuff) are also in Downloads\Convolution_Animation.ipynb

In [ ]:

image = unif_random_config(0.1, 25)
image = np.pad(image, ((1, 1), (1, 1)), 'constant', constant_values=((0, 0), (0, 0)))

y, x = image.shape

geo_found = []

for kernels in orbits:
    particle_no = kernels[0].sum()

    found_mat = np.zeros((y, x))

    orb = []

    for kernel in kernels:
        f = find_geometry(image, kernel, particle_no=particle_no)
        orb.append(f)
    geo_found.append(orb)

fig, ax = plt.subplots(1, 1)

ax.matshow(image)
print(image)

In [ ]:
#CONVOLUTION ANIMATION


from matplotlib.animation import PillowWriter
metadata = dict(title='Movie', artist='codinglikemad')
writer = PillowWriter(fps=16, metadata=metadata)

#with writer.saving(fig, f'CONSTRUCTED_unif_ramsey_fringes.gif', 100):
fig, ax = plt.subplots(1, 1)

ax.matshow(image)

fig = plt.figure(figsize=(15, 5))
spec = gridspec.GridSpec(ncols=2, nrows=1,
                         width_ratios=[2, 1])
convax = fig.add_subplot(spec[0])
configax = fig.add_subplot(spec[1])
temp_img = image.copy()

with writer.saving(fig, f'CONVOLUTION.gif', 100):
    count = 0
    for i, m in zip(orbits, geo_found):
        if count == 73:
            print(i)
            print(m[0].sum())
            print(m[2].sum())
            print(np.nonzero(m[0]))
            print(np.transpose(np.nonzero(m[0])))
            print(m[0])
        convax.xaxis.set_tick_params(labelbottom=False)
        convax.yaxis.set_tick_params(labelleft=False)
        convax.set_xticks([])
        convax.set_yticks([])
        configax.xaxis.set_tick_params(labelbottom=False)
        configax.yaxis.set_tick_params(labelleft=False)
        configax.set_xticks([])
        configax.set_yticks([])

        config = i[-1]
        config = np.pad(config, 1, 'constant', constant_values=((0, 0), (0, 0)))
        colorbar = configax.matshow(config)
        colorbar.set_clim(0, 1)
        comap = convax.matshow(temp_img, cmap='gist_ncar')
        comap.set_clim(0, 1)
        
        
        writer.grab_frame()
        comp_img = temp_img.copy()

        yes = 0
        for o, p in zip(i, m):
            non_zero = np.nonzero(p)
            ty, tx = o.shape
            if ty == 1 and tx == 3:
                for uy, ux in np.transpose(non_zero):
                    temp_img[uy:uy + ty, ux - 1:ux + tx - 1] -= np.rot90(o, k=2)/4
                    yes = 1
            elif ty == 3 and tx == 1:
                for uy, ux in np.transpose(non_zero):
                    temp_img[uy - 1:uy + ty - 1, ux:ux + tx] -= np.rot90(o, k=2)/4
                    yes = 1
            elif ty == 1 and tx == 2:
                for uy, ux in np.transpose(non_zero):
                    temp_img[uy:uy + ty, ux - 1:ux + tx - 1] -= np.rot90(o, k=2)/4
                    yes = 1
            elif ty == 2 and tx == 1:
                for uy, ux in np.transpose(non_zero):
                    temp_img[uy - 1:uy + ty - 1, ux:ux + tx] -= np.rot90(o, k=2)/4
                    yes = 1

            
            else:
                for uy, ux in np.transpose(non_zero):
                    temp_img[uy - 1:uy + ty - 1, ux - 1:ux + tx - 1] -= np.rot90(o, k=2)/4
                    yes = 1
                    
            
        if yes == 1:
            configax.clear()
            colorbar = configax.matshow(config/2)
            colorbar.set_clim(0, 1)
            for i in range(20):
                writer.grab_frame()
            convax.clear()
            convax.xaxis.set_tick_params(labelbottom=False)
            convax.yaxis.set_tick_params(labelleft=False)
            convax.set_xticks([])
            convax.set_yticks([])
            comap = convax.matshow(temp_img, cmap='gist_ncar')
            comap.set_clim(0, 1)
            for i in range(20):
                writer.grab_frame()
            count += 1
        else:
            writer.grab_frame()
        
        convax.clear()
        configax.clear()

                    
print(temp_img)

In [ ]:
# Convolution animation from another (identical?) file


from matplotlib.animation import PillowWriter
metadata = dict(title='Movie', artist='codinglikemad')
writer = PillowWriter(fps=16, metadata=metadata)

#with writer.saving(fig, f'CONSTRUCTED_unif_ramsey_fringes.gif', 100):
fig, ax = plt.subplots(1, 1)

ax.matshow(image)

fig = plt.figure(figsize=(15, 5))
spec = gridspec.GridSpec(ncols=2, nrows=1,
                         width_ratios=[2, 1])
convax = fig.add_subplot(spec[0])
configax = fig.add_subplot(spec[1])
temp_img = image.copy()

with writer.saving(fig, f'CONVOLUTION.gif', 100):
    count = 0
    for i, m in zip(orbits, geo_found):
        if count == 73:
            print(i)
            print(m[0].sum())
            print(m[2].sum())
            print(np.nonzero(m[0]))
            print(np.transpose(np.nonzero(m[0])))
            print(m[0])
        convax.xaxis.set_tick_params(labelbottom=False)
        convax.yaxis.set_tick_params(labelleft=False)
        convax.set_xticks([])
        convax.set_yticks([])
        configax.xaxis.set_tick_params(labelbottom=False)
        configax.yaxis.set_tick_params(labelleft=False)
        configax.set_xticks([])
        configax.set_yticks([])

        config = i[-1]
        config = np.pad(config, 1, 'constant', constant_values=((0, 0), (0, 0)))
        colorbar = configax.matshow(config)
        colorbar.set_clim(0, 1)
        comap = convax.matshow(temp_img, cmap='gist_ncar')
        comap.set_clim(0, 1)
        
        
        writer.grab_frame()
        comp_img = temp_img.copy()

        yes = 0
        for o, p in zip(i, m):
            non_zero = np.nonzero(p)
            ty, tx = o.shape
            if ty == 1 and tx == 3:
                for uy, ux in np.transpose(non_zero):
                    temp_img[uy:uy + ty, ux - 1:ux + tx - 1] -= np.rot90(o, k=2)/4
                    yes = 1
            elif ty == 3 and tx == 1:
                for uy, ux in np.transpose(non_zero):
                    temp_img[uy - 1:uy + ty - 1, ux:ux + tx] -= np.rot90(o, k=2)/4
                    yes = 1
            elif ty == 1 and tx == 2:
                for uy, ux in np.transpose(non_zero):
                    temp_img[uy:uy + ty, ux - 1:ux + tx - 1] -= np.rot90(o, k=2)/4
                    yes = 1
            elif ty == 2 and tx == 1:
                for uy, ux in np.transpose(non_zero):
                    temp_img[uy - 1:uy + ty - 1, ux:ux + tx] -= np.rot90(o, k=2)/4
                    yes = 1

            
            else:
                for uy, ux in np.transpose(non_zero):
                    temp_img[uy - 1:uy + ty - 1, ux - 1:ux + tx - 1] -= np.rot90(o, k=2)/4
                    yes = 1
                    
            
        if yes == 1:
            configax.clear()
            colorbar = configax.matshow(config/2)
            colorbar.set_clim(0, 1)
            for i in range(20):
                writer.grab_frame()
            convax.clear()
            convax.xaxis.set_tick_params(labelbottom=False)
            convax.yaxis.set_tick_params(labelleft=False)
            convax.set_xticks([])
            convax.set_yticks([])
            comap = convax.matshow(temp_img, cmap='gist_ncar')
            comap.set_clim(0, 1)
            for i in range(20):
                writer.grab_frame()
            count += 1
        else:
            writer.grab_frame()
        
        convax.clear()
        configax.clear()

                    
print(temp_img)